# ML-03 - Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smartanilmali234-art/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** - each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring - which one, and why?*

I would frame this lane as a **ranking / scoring** task for content refresh prioritization. The decision is: **which existing pages should an editor review first this week?** Each content item gets a priority score, then the highest-risk and highest-opportunity items move to the top of the queue.

This is better than a plain classification answer because the editor has limited time. A list ordered by priority is more useful than only saying "declining" or "not declining." The observed decline label can still help train or evaluate the score.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

raw = pd.read_csv(DATA_PATH)

lane_summary = pd.DataFrame(
    {
        "task_type": ["Ranking / scoring"],
        "decision": ["Prioritize existing content items for editor refresh"],
        "actor": ["Content strategist or SEO editor"],
        "unit_of_analysis": ["one content item / page"],
        "rows_available": [len(raw)],
        "clients_available": [raw["client_id"].nunique()],
    }
)

lane_summary


,task_type,decision,actor,unit_of_analysis,rows_available,clients_available
0,Ranking / scoring,Prioritize existing content items for editor r...,Content strategist or SEO editor,one content item / page,30000,32


## 2. Target or proxy

*What would you predict? Where does that label come from - observed outcome or a defined rule?*

The near-term proxy target is **whether a content item is declining in search impressions**. In this starter dataset, the available label is `is_declining_label = 1` when `trend_direction == "down"`, which comes from the observed comparison between the most recent 30 days and the previous 30 days.

Because `trend_direction` and `trend_pct` directly define the label, they are **not valid model features**. They are useful only for creating or checking the target. In a production version, I would prefer a future-window target from the daily warehouse, such as "declines in the next 30 days," so features are measured before the outcome.


In [2]:
target = (raw["trend_direction"].eq("down")).astype(int).rename("is_declining_label")

label_check = pd.DataFrame(
    {
        "rows": [len(raw)],
        "declining_items": [int(target.sum())],
        "declining_rate": [target.mean()],
        "label_source": ["trend_direction == 'down'"],
        "leakage_columns_excluded_from_features": ["trend_direction, trend_pct"],
    }
)

label_check


,rows,declining_items,declining_rate,label_source,leakage_columns_excluded_from_features
0,30000,16262,0.542067,trend_direction == 'down',"trend_direction, trend_pct"


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I would use **precision@K** for the ranking queue. If an editor can refresh only the top 100 items, the useful question is: **what share of those 100 are truly declining?**

A defensible first target is to beat the dataset base rate by a clear margin. In this notebook, I compare a simple opportunity score against the overall decline rate. For a future trained model, I would report precision@50 or precision@100 on a client-held-out split, because the system must generalize to clients whose pages were not used in training.


In [3]:
def precision_at_k(y_true, score, k=100):
    ranked = pd.DataFrame({"y": y_true, "score": score}).sort_values("score", ascending=False)
    return ranked.head(k)["y"].mean()

# A transparent baseline: prioritize pages with many impressions and older updates.
# This is not the final model; it gives the assignment a measurable starting point.
baseline_score = (
    np.log1p(raw["impressions_90d"])
    + 0.5 * np.log1p(raw["days_since_last_update"])
    + raw["avg_position"].replace(0, np.nan).fillna(raw["avg_position"].replace(0, np.nan).median()) / 50
)

metric_summary = pd.DataFrame(
    {
        "metric": ["precision@100"],
        "base_decline_rate": [target.mean()],
        "baseline_precision_at_100": [precision_at_k(target, baseline_score, k=100)],
        "success_definition": ["future model beats base rate and simple baseline on client-held-out data"],
    }
)

metric_summary


,metric,base_decline_rate,baseline_precision_at_100,success_definition
0,precision@100,0.542067,0.5,future model beats base rate and simple baseli...


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row is **one pseudonymized content item**. The dataframe below keeps only safe aggregate columns that an editor could use for prioritization. It does not show `content_id`, `client_id`, private URLs, client names, or raw queries.

For modeling, `content_id` and `client_id` should be used only for grouping, joining, and validation splits, not as predictive features.


In [4]:
safe_feature_columns = [
    "content_type",
    "main_intent",
    "search_volume",
    "competition",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "ai_traffic_pct",
]

lane_df = raw[safe_feature_columns].copy()
lane_df["avg_position_no_data"] = lane_df["avg_position"].eq(0)
lane_df["has_word_count"] = lane_df["word_count"].notna()
lane_df["is_declining_label"] = target

print(f"lane_df shape: {lane_df.shape}")
lane_df.head(10)


lane_df shape: (30000, 17)


,content_type,main_intent,search_volume,competition,word_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,ai_traffic_pct,avg_position_no_data,has_word_count,is_declining_label
0,keyword article,transactional,10.0,0.67,3221.0,187,20,3803,29,17,0.76,10.6,5.88,0.0,False,True,1
1,keyword article,informational,90.0,0.01,2481.0,445,25,15320,7,9,0.05,20.3,0.00,0.0,False,True,1
2,keyword article,informational,0.0,0.00,3515.0,141,20,12581,11,11,0.09,36.5,0.00,0.0,False,True,1
3,keyword article,commercial,10.0,0.00,NaN,463,22,11751,58,78,0.49,6.2,1.28,0.0,False,False,0
4,keyword article,informational,0.0,0.00,2803.0,263,14,19140,24,145,0.13,44.0,0.00,0.0,False,True,1
5,keyword article,transactional,720.0,1.00,3080.0,147,20,3970,1,5,0.03,8.5,0.00,0.0,False,True,1
6,keyword article,informational,0.0,0.00,3059.0,90,20,20,0,1,0.00,7.0,0.00,0.0,False,True,1
7,keyword article,commercial,590.0,0.44,NaN,445,22,1724,1,28,0.06,21.2,3.57,0.0,False,False,0
8,keyword article,informational,0.0,0.00,3807.0,90,20,32574,29,68,0.09,46.0,5.88,0.0,False,True,1
9,keyword article,informational,0.0,0.00,NaN,257,104,1240,2,3,0.16,4.9,0.00,0.0,False,False,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "refresh old pages with low CTR" is too brittle because decline risk depends on several signals at the same time: search volume, impressions, position, engagement, content age, intent, content type, and missing measurement patterns. Some rates are percentages stored as x100 values, `avg_position = 0` means no data, and missing keyword/content fields are systematic by content type.

ML can help because it can combine many weak signals into a ranked queue. The output should still be treated as **decision support**, not an automatic command to rewrite content.


In [5]:
# Privacy-safe diagnostics that explain why a single if-statement is weak.
missing_by_type = (
    raw.groupby("content_type", dropna=False)
    .agg(
        rows=("content_type", "size"),
        word_count_missing_rate=("word_count", lambda s: s.isna().mean()),
        search_volume_missing_rate=("search_volume", lambda s: s.isna().mean()),
        decline_rate=("trend_direction", lambda s: s.eq("down").mean()),
        median_impressions_90d=("impressions_90d", "median"),
        median_ctr_pct=("ctr", "median"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

rate_sanity = pd.DataFrame(
    {
        "check": [
            "ctr is stored as percent, not fraction",
            "avg_position zero means no position data",
            "scroll_rate can exceed 100",
            "ai_traffic_pct can exceed 100",
        ],
        "value": [
            f"median ctr = {raw['ctr'].median():.2f}%",
            int(raw["avg_position"].eq(0).sum()),
            int(raw["scroll_rate"].gt(100).sum()),
            int(raw["ai_traffic_pct"].gt(100).sum()),
        ],
    }
)

print("Missingness and decline differ by content type:")
display(missing_by_type)

print("Data gotchas to handle before modeling:")
display(rate_sanity)


Missingness and decline differ by content type:


,content_type,rows,word_count_missing_rate,search_volume_missing_rate,decline_rate,median_impressions_90d,median_ctr_pct
2,keyword article,27207,0.282979,0.013673,0.560959,955.0,0.09
1,feedly article,2096,0.000000,1.000000,0.286737,4.0,0.00
0,comparison article,697,0.000000,0.000000,0.572453,107.0,0.00


Data gotchas to handle before modeling:


,check,value
0,"ctr is stored as percent, not fraction",median ctr = 0.07%
1,avg_position zero means no position data,1205
2,scroll_rate can exceed 100,119
3,ai_traffic_pct can exceed 100,23


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.


In [6]:
# Programmatic self-check for this assignment.
# This cell is self-contained, so it works even if you run it after restarting the kernel.
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

raw = pd.read_csv(DATA_PATH)
target = (raw["trend_direction"].eq("down")).astype(int).rename("is_declining_label")

safe_feature_columns = [
    "content_type",
    "main_intent",
    "search_volume",
    "competition",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "ai_traffic_pct",
]
lane_df = raw[safe_feature_columns].copy()
lane_df["avg_position_no_data"] = lane_df["avg_position"].eq(0)
lane_df["has_word_count"] = lane_df["word_count"].notna()
lane_df["is_declining_label"] = target

def precision_at_k(y_true, score, k=100):
    ranked = pd.DataFrame({"y": y_true, "score": score}).sort_values("score", ascending=False)
    return ranked.head(k)["y"].mean()

baseline_score = (
    np.log1p(raw["impressions_90d"])
    + 0.5 * np.log1p(raw["days_since_last_update"])
    + raw["avg_position"].replace(0, np.nan).fillna(raw["avg_position"].replace(0, np.nan).median()) / 50
)
metric_summary = pd.DataFrame(
    {
        "metric": ["precision@100"],
        "base_decline_rate": [target.mean()],
        "baseline_precision_at_100": [precision_at_k(target, baseline_score, k=100)],
    }
)

rate_sanity = pd.DataFrame(
    {
        "check": [
            "ctr is stored as percent, not fraction",
            "avg_position zero means no position data",
            "scroll_rate can exceed 100",
            "ai_traffic_pct can exceed 100",
        ],
        "value": [
            f"median ctr = {raw['ctr'].median():.2f}%",
            int(raw["avg_position"].eq(0).sum()),
            int(raw["scroll_rate"].gt(100).sum()),
            int(raw["ai_traffic_pct"].gt(100).sum()),
        ],
    }
)

checks = {
    "data_loaded_30000_rows": len(raw) == 30_000,
    "one_row_per_content_item": raw["content_id"].is_unique,
    "target_matches_observed_decline_proxy": target.equals(raw["trend_direction"].eq("down").astype(int).rename("is_declining_label")),
    "leakage_columns_not_in_lane_df": not {"trend_direction", "trend_pct"}.intersection(lane_df.columns),
    "private_ids_not_displayed_in_lane_df": not {"content_id", "client_id"}.intersection(lane_df.columns),
    "metric_is_precision_at_100": metric_summary.loc[0, "metric"] == "precision@100",
    "precision_at_100_is_valid_probability": 0 <= metric_summary.loc[0, "baseline_precision_at_100"] <= 1,
    "safe_missingness_flag_added": "has_word_count" in lane_df.columns,
    "avg_position_no_data_flag_added": "avg_position_no_data" in lane_df.columns,
    "data_gotcha_summary_created": {"check", "value"}.issubset(rate_sanity.columns),
}

self_check = pd.DataFrame({"check": list(checks.keys()), "passed": list(checks.values())})

if not self_check["passed"].all():
    failed = self_check.loc[~self_check["passed"], "check"].tolist()
    raise AssertionError(f"Self-check failed: {failed}")

print("All programmatic self-checks passed.")
self_check


All programmatic self-checks passed.


,check,passed
0,data_loaded_30000_rows,True
1,one_row_per_content_item,True
2,target_matches_observed_decline_proxy,True
3,leakage_columns_not_in_lane_df,True
4,private_ids_not_displayed_in_lane_df,True
5,metric_is_precision_at_100,True
6,precision_at_100_is_valid_probability,True
7,safe_missingness_flag_added,True
8,avg_position_no_data_flag_added,True
9,data_gotcha_summary_created,True
